# COMP8240 VADER final project

Run this notebook from the project root after installing requirements. Observed results from 4 November 2026 are in `results/`; this new notebook has not itself been executed. Human annotation is still pending. See the README for the CLI workflow.

In [ ]:
from pathlib import Path
import sys, subprocess, json
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir() and (ROOT.parent / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src/run_experiments.py').exists(), 'Open from the extracted project root.'
print('Project:', ROOT)


## Reproduce pinned-source experiments
This downloads the source resources, verifies checksums, scores both VADER versions, and exports results. Thresholds remain fixed.

In [ ]:
subprocess.run([sys.executable, str(ROOT/'src/run_experiments.py')], cwd=ROOT, check=True)


In [ ]:
import pandas as pd
pd.read_csv(ROOT/'results/original-data-results.csv')


In [ ]:
bench = json.loads((ROOT/'results/benchmark-results.json').read_text())
pd.DataFrame([{'split':k, 'implementation':v, 'n':r['n'], 'accuracy':r[v]['accuracy'], 'macro_f1':r[v]['macro_f1']} for k,r in bench.items() for v in ['archived','maintained']])


## Rescore preserved within-session judge responses
These are assistant judgements with disclosed limits, not independent human ratings or a fresh API run.

In [ ]:
subprocess.run([sys.executable, str(ROOT/'src/evaluate_judge.py')], cwd=ROOT, check=True)


## Human annotation
Ask two real readers to complete their independent sheets using the guidelines. Blank sheets produce PENDING and no human metrics. Do not replace them with AI labels.

In [ ]:
result = subprocess.run([sys.executable, str(ROOT/'src/evaluate_human.py')], cwd=ROOT)
print('Human-evaluation return code:', result.returncode, '(2 means pending)')


In [ ]:
for script in ['make_figures.py','export_report_tables.py','check_project.py']:
    subprocess.run([sys.executable, str(ROOT/'src'/script)], cwd=ROOT, check=True)


## Report
Compile `report/main.tex` with latexmk or Overleaf. Keep the review-draft notice until real human results and the public repository URL have been added. Confirm the remaining course requirements in `docs/SUBMISSION_CHECKLIST.md`.